# Enterprise Data Discovery Notebook

## Sprint 1 - Discovery & Data Understanding

This notebook aims to automatically discover, analyze, and document the available datasets before building the data platform.

## 1. Objectives

- Automatically discover available datasets
- Analyze file structure and metadata
- Generate an initial data inventory
- Perform technical data profiling
- Identify potential primary and foreign key relationships
- Prepare documentation for future Bronze, Silver, and Gold layers

## 2. Environment Setup

This section imports the required Python libraries and configures the project paths.

Using relative paths ensures that the notebook remains portable across local development environments and Microsoft Fabric notebooks.

In [56]:
# ------------------------------------------------------------------
# Import required libraries
#
# pathlib : Handles file system paths in a platform-independent way.
# pandas  : Loads and analyzes tabular datasets.
# numpy   : Supports numerical operations used throughout the project.
# ------------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# Configure project directories
#
# All project paths are defined using relative locations to ensure
# the notebook can run on different environments without modifying
# the source code.
# ------------------------------------------------------------------

PROJECT_PATH = Path("..")

DATASET_PATH = PROJECT_PATH / "datasets"

ARTIFACT_PATH = PROJECT_PATH / "artifacts"



## 3. Discover Available Source Datasets

Before analyzing the data, the notebook automatically discovers every available dataset within the project.

The datasets are organized by business domain following Data Mesh principles. This approach allows the solution to scale without modifying the source code whenever new datasets are added.

At this stage, no data is loaded into memory. The objective is simply to inventory the available source systems.

In [57]:
# ------------------------------------------------------------------
# Discover Available Source Datasets
#
# Scan the datasets directory recursively and identify every CSV file
# organized by business domain.
#
# This approach removes the need to hardcode dataset names and allows
# the notebook to automatically detect new source systems.
# ------------------------------------------------------------------

# Search recursively for all CSV files
csv_files = sorted(DATASET_PATH.rglob("*.csv"))

print(f"Datasets discovered: {len(csv_files)}\n")

for file in csv_files:
    print(file.relative_to(DATASET_PATH))

Datasets discovered: 9

commerce\raw\olist_order_items_dataset.csv
commerce\raw\olist_order_payments_dataset.csv
commerce\raw\olist_orders_dataset.csv
commerce\raw\olist_products_dataset.csv
commerce\raw\olist_sellers_dataset.csv
commerce\raw\product_category_name_translation.csv
custommer\raw\olist_customers_dataset.csv
custommer\raw\olist_order_reviews_dataset.csv
delivery\raw\olist_geolocation_dataset.csv


## 4. Build the Enterprise Data Inventory

Once the available datasets have been discovered, the next step is to build a technical inventory.

The inventory provides a high-level overview of the available source systems, including:

- Business domain
- Dataset name
- File size
- Number of records
- Number of columns

This inventory serves as the foundation for data discovery, source system documentation, and future ingestion pipelines.

In [58]:
# ------------------------------------------------------------------
# Build Enterprise Data Inventory
#
# Create a technical inventory describing every dataset available
# in the project.
#
# This inventory will later support documentation, ingestion planning,
# and data governance activities.
# ------------------------------------------------------------------

inventory = []

for file in csv_files:

    df = pd.read_csv(file)

    inventory.append({
        "Domain": file.parts[-3].capitalize(),
        "Dataset": file.stem,
        "Rows": len(df),
        "Columns": len(df.columns),
        "File Size (MB)": round(file.stat().st_size / (1024 * 1024), 2),
        "Format": file.suffix.replace(".", "").upper(),
        "Status": "Ready"
    })

enterprise_data_inventory = pd.DataFrame(inventory)

enterprise_data_inventory

,Domain,Dataset,Rows,Columns,File Size (MB),Format,Status
0,Commerce,olist_order_items_dataset,112650,7,14.72,CSV,Ready
1,Commerce,olist_order_payments_dataset,103886,5,5.51,CSV,Ready
2,Commerce,olist_orders_dataset,99441,8,16.84,CSV,Ready
3,Commerce,olist_products_dataset,32951,9,2.27,CSV,Ready
4,Commerce,olist_sellers_dataset,3095,4,0.17,CSV,Ready
5,Commerce,product_category_name_translation,71,2,0.00,CSV,Ready
6,Custommer,olist_customers_dataset,99441,5,8.62,CSV,Ready
7,Custommer,olist_order_reviews_dataset,99224,7,13.78,CSV,Ready
8,Delivery,olist_geolocation_dataset,1000163,5,58.44,CSV,Ready


## 5. Build the Data Profiling Report

Once the enterprise data inventory has been created, the next step is to analyze the internal structure and quality of each dataset.

The profiling report provides a detailed overview of the available data, including:

- Column names
- Data types
- Null values
- Unique values
- Basic statistics
- Data completeness

This report helps identify data quality issues, potential business keys, and transformation requirements before designing the Lakehouse architecture and data models.

In [59]:
# ------------------------------------------------------------------
# Build Data Profiling Report
#
# Analyze the structure and quality of every dataset discovered
# during the inventory phase.
#
# This profiling report will later support data quality checks,
# transformation design, and Lakehouse modeling decisions.
# ------------------------------------------------------------------

profiling = []

for file in csv_files:

    df = pd.read_csv(file)

    for column in df.columns:

        profiling.append({
            "Domain": file.parts[-3].capitalize(),
            "Dataset": file.stem,
            "Column": column,
            "Data Type": str(df[column].dtype),
            "Rows": len(df),
            "Null Count": int(df[column].isna().sum()),
            "Null Percentage (%)": round(
                (df[column].isna().sum() / len(df)) * 100, 2
            ),
            "Unique Values": int(df[column].nunique()),
            "Unique Percentage (%)": round(
                (df[column].nunique() / len(df)) * 100, 2
            )
        })

enterprise_data_profile = pd.DataFrame(profiling)

enterprise_data_profile

,Domain,Dataset,Column,Data Type,Rows,Null Count,Null Percentage (%),Unique Values,Unique Percentage (%)
0,Commerce,olist_order_items_dataset,order_id,str,112650,0,0.00,98666,87.59
1,Commerce,olist_order_items_dataset,order_item_id,int64,112650,0,0.00,21,0.02
2,Commerce,olist_order_items_dataset,product_id,str,112650,0,0.00,32951,29.25
3,Commerce,olist_order_items_dataset,seller_id,str,112650,0,0.00,3095,2.75
4,Commerce,olist_order_items_dataset,shipping_limit_date,str,112650,0,0.00,93318,82.84
5,Commerce,olist_order_items_dataset,price,float64,112650,0,0.00,5968,5.30
6,Commerce,olist_order_items_dataset,freight_value,float64,112650,0,0.00,6999,6.21
7,Commerce,olist_order_payments_dataset,order_id,str,103886,0,0.00,99440,95.72
8,Commerce,olist_order_payments_dataset,payment_sequential,int64,103886,0,0.00,29,0.03
9,Commerce,olist_order_payments_dataset,payment_type,str,103886,0,0.00,5,0.00


## 6. Discover Dataset Relationships

Once the data profiling phase has been completed, the next step is to identify relationships between datasets.

The relationship discovery process analyzes potential:

- Primary Keys (PK)
- Foreign Keys (FK)
- One-to-Many relationships
- Many-to-One relationships
- Entity dependencies

This analysis provides the foundation for designing the enterprise data model, including the future Lakehouse architecture and dimensional models.

In [61]:
# ------------------------------------------------------------------
# Define Semantic Column Mapping
#
# Map equivalent business attributes that use different physical
# column names across datasets.
# ------------------------------------------------------------------

semantic_mapping = {

    "customer_id": [
        "customer_id"
    ],

    "order_id": [
        "order_id"
    ],

    "product_id": [
        "product_id"
    ],

    "seller_id": [
        "seller_id"
    ],

    "postal_code": [
        "customer_zip_code_prefix",
        "seller_zip_code_prefix",
        "geolocation_zip_code_prefix"
    ]

}

In [62]:
# ------------------------------------------------------------------
# Get Business Attribute
#
# Return the business attribute associated with a physical column.
# ------------------------------------------------------------------

def get_business_attribute(column_name):

    for attribute, columns in semantic_mapping.items():

        if column_name in columns:
            return attribute

    return None

In [63]:
# ------------------------------------------------------------------
# Load Datasets
#
# Load all discovered datasets into memory.
# ------------------------------------------------------------------

datasets = {}

for file in csv_files:

    datasets[file.stem] = pd.read_csv(file)

In [64]:
# ------------------------------------------------------------------
# Discover Dataset Relationships
#
# Identify potential relationships between datasets by analyzing
# common column names and value overlap.
#
# This discovery process will support future PK/FK definitions,
# dimensional modeling, and Lakehouse architecture decisions.
# ------------------------------------------------------------------

relationships = []

for source_name, source_df in datasets.items():

    for target_name, target_df in datasets.items():

        if source_name == target_name:
            continue

        common_columns = set(source_df.columns).intersection(
            set(target_df.columns)
        )

        for column in common_columns:

            source_values = set(
                source_df[column]
                .dropna()
                .astype(str)
                .unique()
            )

            target_values = set(
                target_df[column]
                .dropna()
                .astype(str)
                .unique()
            )

            if len(source_values) == 0:
                continue

            overlap = (
                len(source_values.intersection(target_values))
                / len(source_values)
            )

            if overlap > 0.5:

                relationships.append({

                    "Relationship Type": "Structural",

                    "Source Dataset": source_name,

                    "Source Column": column,

                    "Target Dataset": target_name,

                    "Target Column": column,

                    "Business Attribute": get_business_attribute(column),

                    "Match Percentage (%)": round(
                        overlap * 100,
                        2
                    )

                })

In [65]:
# ------------------------------------------------------------------
# Discover Semantic Relationships
#
# Identify relationships based on equivalent business attributes.
# ------------------------------------------------------------------

for source_name, source_df in datasets.items():

    for target_name, target_df in datasets.items():

        if source_name == target_name:
            continue

        for source_column in source_df.columns:

            source_attribute = get_business_attribute(source_column)

            if source_attribute is None:
                continue

            for target_column in target_df.columns:

                target_attribute = get_business_attribute(target_column)

                if source_attribute != target_attribute:
                    continue

                if source_column == target_column:
                    continue

                source_values = set(
                    source_df[source_column]
                    .dropna()
                    .astype(str)
                    .unique()
                )

                target_values = set(
                    target_df[target_column]
                    .dropna()
                    .astype(str)
                    .unique()
                )

                if len(source_values) == 0:
                    continue

                overlap = (
                    len(source_values.intersection(target_values))
                    / len(source_values)
                )

                if overlap > 0.5:

                    relationships.append({

                        "Relationship Type": "Semantic",

                        "Source Dataset": source_name,

                        "Source Column": source_column,

                        "Target Dataset": target_name,

                        "Target Column": target_column,

                        "Business Attribute": source_attribute,

                        "Match Percentage (%)": round(
                            overlap * 100,
                            2
                        )

                    })

In [69]:
# ------------------------------------------------------------------
# Build Enterprise Relationship Discovery
#
# Create the final relationship catalog.
# ------------------------------------------------------------------

enterprise_relationship_discovery = (
    pd.DataFrame(relationships)
    .sort_values(
        by=[
            "Relationship Type",
            "Source Dataset",
            "Target Dataset",
            "Source Column"
        ]
    )
    .reset_index(drop=True)
)

enterprise_relationship_discovery


,Relationship Type,Source Dataset,Source Column,Target Dataset,Target Column,Business Attribute,Match Percentage (%)
0,Semantic,olist_customers_dataset,customer_zip_code_prefix,olist_geolocation_dataset,geolocation_zip_code_prefix,postal_code,98.95
1,Semantic,olist_geolocation_dataset,geolocation_zip_code_prefix,olist_customers_dataset,customer_zip_code_prefix,postal_code,78.03
2,Semantic,olist_sellers_dataset,seller_zip_code_prefix,olist_customers_dataset,customer_zip_code_prefix,postal_code,96.26
3,Semantic,olist_sellers_dataset,seller_zip_code_prefix,olist_geolocation_dataset,geolocation_zip_code_prefix,postal_code,99.69
4,Structural,olist_customers_dataset,customer_id,olist_orders_dataset,customer_id,customer_id,100.00
5,Structural,olist_order_items_dataset,order_id,olist_order_payments_dataset,order_id,order_id,100.00
6,Structural,olist_order_items_dataset,order_id,olist_order_reviews_dataset,order_id,order_id,99.24
7,Structural,olist_order_items_dataset,order_id,olist_orders_dataset,order_id,order_id,100.00
8,Structural,olist_order_items_dataset,product_id,olist_products_dataset,product_id,product_id,100.00
9,Structural,olist_order_items_dataset,seller_id,olist_sellers_dataset,seller_id,seller_id,100.00
